# NN · Modelo: _[NOMBRE DEL MODELO]_

**Proyecto Final — Machine Learning y Deep Learning**

> 📋 **Plantilla.** Copia este notebook, renumera/renombra y **rellena los `TODO`**. Mantén las 7 secciones y deja la **presentación inline** (visible). Importa de `src` solo el **contrato** (datos, preprocesado, config, definición del modelo). Convención en [`README.md`](README.md).

## 1. Cómo funciona

_TODO: explica en lenguaje sencillo cómo funciona el algoritmo._

## 2. Los datos

Usamos el **preprocesado compartido** (`src/preprocessing.py`). EDA en [`01_eda.ipynb`](01_eda.ipynb).

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)
# Importamos solo el CONTRATO del pipeline (mismos datos y preprocesado que producción):
from ml_hotel_cancellations import config
from ml_hotel_cancellations.ml.data_loader import load_and_prepare
from ml_hotel_cancellations.ml.preprocessing import build_preprocessor

X_train, X_test, y_train, y_test = load_and_prepare()
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

def evaluar(pipe, X_test, y_test, etiqueta='Modelo'):
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    m = dict(accuracy=accuracy_score(y_test, pred), precision=precision_score(y_test, pred),
             recall=recall_score(y_test, pred), f1=f1_score(y_test, pred),
             roc_auc=roc_auc_score(y_test, proba))
    print(f"{etiqueta:12s} | " + ' | '.join(f'{k}={v:.4f}' for k, v in m.items()))
    return m['roc_auc']

## 3. Los hiperparámetros: ¿qué controla cada uno?

_TODO: tabla._

| Hiperparámetro | Qué controla |
|---|---|
| `...` | _TODO_ |

## 4. Entrenamiento y evaluación (parámetros base)

In [ ]:
# TODO: importa y construye tu modelo (hiperparámetros desde config si existen)
from sklearn.linear_model import LogisticRegression  # <-- cambia por tu modelo
modelo = Pipeline([('preprocessor', build_preprocessor()),
                   ('model', LogisticRegression(**config.LOGISTIC_REGRESSION_PARAMS))])
modelo.fit(X_train, y_train)
auc_base = evaluar(modelo, X_test, y_test, 'Base')

In [ ]:
# Matriz de confusión y curva ROC
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, ax=ax1,
    display_labels=['No cancela', 'Cancela'], cmap='Blues', colorbar=False)
ax1.set_title('Matriz de confusión')
RocCurveDisplay.from_estimator(modelo, X_test, y_test, ax=ax2)
ax2.plot([0, 1], [0, 1], '--', color='gray', alpha=0.7); ax2.set_title('Curva ROC')
plt.tight_layout(); plt.show()

## 5. Visualización del modelo

_TODO: visualiza la estructura/parámetros de tu modelo (queda **inline**, visible)._

In [ ]:
# TODO: dibuja tu modelo. Ejemplos según el tipo:
#  - logística: barras de modelo.named_steps['model'].coef_
#  - árbol / RF: from sklearn.tree import plot_tree; plot_tree(...)
#  - XGBoost: barras de modelo.named_steps['model'].feature_importances_
#  - red: esquema de capas con matplotlib
nombres = modelo.named_steps['preprocessor'].get_feature_names_out()
# ...

## 6. Optimización de hiperparámetros

_TODO: GridSearchCV (espacios pequeños) o RandomizedSearchCV (grandes), optimizando ROC-AUC por CV._

In [ ]:
from sklearn.model_selection import GridSearchCV  # o RandomizedSearchCV
base_pipe = Pipeline([('preprocessor', build_preprocessor()),
                      ('model', LogisticRegression(max_iter=1000, random_state=config.RANDOM_STATE))])
busqueda = GridSearchCV(base_pipe, config.LOGISTIC_REGRESSION_GRID,
                        scoring='roc_auc', cv=config.TUNING_CV_FOLDS, n_jobs=-1)
busqueda.fit(X_train, y_train)
print('Mejores:', busqueda.best_params_); print(f'ROC-AUC CV: {busqueda.best_score_:.4f}')

In [ ]:
auc_tuned = evaluar(busqueda.best_estimator_, X_test, y_test, 'Optimizado')
print(f'\nROC-AUC en test:  base {auc_base:.4f}  ->  optimizado {auc_tuned:.4f}')

## 7. Resultado final y cuándo usar este modelo

_TODO._